# Pregunta 1 — Clasificacion multietiqueta de anomalias de rodilla en MRI

**RSNA Knee Abnormality Detection.** Este cuaderno *orquesta* y *audita* el
pipeline de la Pregunta 1: documenta el comando unico reproducible, y carga los
JSON de resultados ya versionados para reconstruir las tablas clave del informe
y mostrar los mapas Grad-CAM, **sin necesidad de descargar el dataset**.

> La ejecucion COMPLETA del pipeline (entrenamiento, preprocesamiento, Grad-CAM)
> requiere el dataset RSNA Knee Abnormality Detection. Este cuaderno solo *lee*
> los resultados que el pipeline ya escribio, de modo que corre de principio a
> fin con lo que hay en el repositorio.

## El pipeline en seis pasos

| Paso | Que hace | Entregables |
|---|---|---|
| 1 — Etiquetas debiles | 4 349 reportes en 9 idiomas -> 12 etiquetas binarias (reglas + NegEx + traduccion NLLB) | #1, #7 |
| 2 — Preprocesamiento | ZIP de 570 GB -> muestra estratificada -> tensores 2.5D cacheados | — |
| 3 — Modelos | Inception desde cero + ResNet-50 ImageNet, en condiciones identicas | #2 |
| 4 — Evaluacion | AUC-ROC macro + desglose por etiqueta + bootstrap sobre los dorados | #3 |
| 5 — Ablacion / Grad-CAM | 7 comparaciones pareadas, 3 mapas de activacion, desbalance | #4, #5, #6 |
| 6 — Informe | Informe NeurIPS (PDF) + este repositorio reproducible | transversal |


## El comando unico reproducible

Desde un entorno limpio, de extremo a extremo:

```bash
# 1. Entorno virtual
python -m venv .venv
# Windows:        .venv\Scripts\activate
# Linux / macOS:  source .venv/bin/activate

# 2. Dependencias de la Pregunta 1
pip install -r pregunta1/requirements.txt
# GPU (CUDA 12.8):
# pip install "torch>=2.11" "torchvision>=0.26" --index-url https://download.pytorch.org/whl/cu128

# 3. Pipeline completo (idempotente: salta lo ya hecho)
python pregunta1/paso6_informe/ejecutar_todo.py
```

La celda siguiente *lista* las etapas del pipeline sin ejecutar ninguna
(`--listar`), para ver que haria el comando unico.


In [ ]:
import subprocess, sys, pathlib

# Raiz del repo = carpeta que contiene a pregunta1/. Este cuaderno vive en
# pregunta1/, asi que la raiz es su directorio padre.
BASE = pathlib.Path.cwd()
if BASE.name != "pregunta1" and (BASE / "pregunta1").is_dir():
    PREG = BASE / "pregunta1"
    RAIZ = BASE
else:
    PREG = BASE if BASE.name == "pregunta1" else BASE
    RAIZ = PREG.parent
print("pregunta1:", PREG)

orquestador = PREG / "paso6_informe" / "ejecutar_todo.py"
try:
    salida = subprocess.run([sys.executable, str(orquestador), "--listar"],
                            cwd=RAIZ, capture_output=True, text=True, timeout=120)
    print(salida.stdout or salida.stderr)
except Exception as e:
    print("No se pudo listar las etapas (requiere el repo completo):", e)


## Cargar los resultados (sin dataset)

Todo lo que sigue lee unicamente los JSON que el pipeline ya escribio. No
importa PyTorch ni toca el dataset.


In [ ]:
import json, pathlib

def cargar(ruta_relativa):
    ruta = PREG / ruta_relativa
    with ruta.open(encoding="utf-8") as fh:
        return json.load(fh)

comparaciones = cargar("paso4_evaluacion/comparaciones.json")
por_caras     = cargar("paso4_evaluacion/evaluacion_por_caras.json")
detalle       = cargar("paso4_evaluacion/detalle_por_etiqueta.json")
estrategias   = cargar("paso1_etiquetas_debiles/comparar_estrategias.json")
print("JSON cargados:",
      len(comparaciones), "comparaciones,",
      len(detalle["etiquetas"]), "etiquetas en el desglose")


### Entregable #1 — extraccion de etiquetas debiles

Cuatro estrategias de traduccion medidas contra los 58 estudios dorados. El
esquema **hibrido** (traducir salvo ingles y espanol) es el que produjo las
etiquetas con que se entrenaron los modelos: ruido de entrenamiento 25.7 %.


In [ ]:
print(f"{'estrategia':34} {'acuerdo':>8} {'F1 macro':>9} {'ruido':>7}")
print('-' * 60)
for e in estrategias['estrategias']:
    print(f"{e['estrategia']:34} {e['acuerdo_macro']:>7.1f}% "
          f"{e['f1_macro']:>9.3f} {e['ruido_macro']:>6.1f}%")
print('\nmejor:', estrategias['mejor'])


### Entregables #2 y #3 — AUC-ROC macro por configuracion

Dos regimenes de datos distintos (no comparables entre si). El modelo del
analisis detallado es **ResNet-50 con promedio** (0.708); la configuracion E
alcanza el mayor AUC puntual (0.761), con un intervalo que se solapa con el de B.


In [ ]:
def busca_auc(nombre, campo_a='a', campo_b='b', campo_auc_a='auc_a', campo_auc_b='auc_b'):
    for c in comparaciones:
        if c[campo_a] == nombre:
            return c[campo_auc_a]
        if c[campo_b] == nombre:
            return c[campo_auc_b]
    return None

filas = [
    ("A  Inception desde cero (576 est.)",  busca_auc('inception_bce_atencion'), None),
    ("B  ResNet-50 ImageNet (576 est.)",    busca_auc('resnet50_bce_atencion'),  None),
    ("B + promedio (576 est.)  [detallado]", busca_auc('resnet50_promedio'),     [0.662, 0.755]),
    ("C  Inception, 1 modelo de 12 (4407)", por_caras['INCEPTION modelo unico de 12']['macro'],
                                            por_caras['INCEPTION modelo unico de 12']['ic95']),
    ("D  Inception por caras (4407)",       por_caras['INCEPTION por caras']['macro'],
                                            por_caras['INCEPTION por caras']['ic95']),
    ("E  ResNet-50 por caras (4407)  [max]", por_caras['RESNET-50 por caras']['macro'],
                                            por_caras['RESNET-50 por caras']['ic95']),
]
print(f"{'configuracion':40} {'AUC macro':>10}  IC95%")
print('-' * 72)
for nombre, auc, ic in filas:
    ic_txt = f"[{ic[0]:.3f}, {ic[1]:.3f}]" if ic else ""
    print(f"{nombre:40} {auc:>10.3f}  {ic_txt}")


### Entregable #3 — desglose por etiqueta (modelo ResNet-50 con promedio, umbral 0.20)

Un asterisco marca las etiquetas cuyo IC95 % incluye 0.5 (no se distinguen del azar).


In [ ]:
print(f"{'etiqueta':18} {'pos':>4} {'AUC':>7} {'IC95%':>18} {'F1':>6}")
print('-' * 58)
for d in detalle['etiquetas']:
    marca = '*' if d.get('cruza_azar') else ' '
    ic = f"[{d['bajo']:.2f}, {d['alto']:.2f}]"
    print(f"{d['etiqueta']:18} {d['positivos']:>4} {d['auc']:>7.3f} "
          f"{ic:>18} {d['f1']:>6.3f} {marca}")
cruzan = sum(1 for d in detalle['etiquetas'] if d.get('cruza_azar'))
print(f"\n{cruzan} de {len(detalle['etiquetas'])} etiquetas cruzan el azar")


### Entregable #4 — ablacion (bootstrap pareado sobre los dorados)

Solo 2 de 7 diferencias son significativas (IC que no cruza cero), y las dos
son decisiones sobre los datos (ImageNet, plano), no sobre la arquitectura.


In [ ]:
print(f"{'comparacion':40} {'dAUC':>7} {'IC95%':>20} {'signif':>7}")
print('-' * 78)
for c in comparaciones:
    ic = f"[{c['bajo']:+.3f}, {c['alto']:+.3f}]"
    sig = 'si' if c['significativa'] else 'no'
    print(f"{c['descripcion'][:40]:40} {c['delta']:>+7.3f} {ic:>20} {sig:>7}")


### Entregable #5 — Grad-CAM

Mapas de activacion para tres hallazgos, sobre ResNet-50 con promedio.
Derrame y ligamento cruzado anterior se apoyan en regiones anatomicamente
plausibles; el menisco medial acierta apoyandose fuera de la articulacion.


In [ ]:
from IPython.display import Image, display

figs = PREG / "paso5_ablacion_gradcam" / "figuras"
# El informe incrusta estas tres; si existen en informe/figuras, usarlas.
alt = PREG / "paso6_informe" / "informe" / "figuras"
nombres = ["gradcam_Effusion.png", "gradcam_ACL.png", "gradcam_Medial_Meniscus.png"]
for n in nombres:
    ruta = alt / n if (alt / n).is_file() else figs / n
    if ruta.is_file():
        print(n)
        display(Image(filename=str(ruta)))
    else:
        print("(no encontrada:", n, ")")


---

Para regenerar cualquiera de estos numeros desde cero (con el dataset RSNA
disponible), correr el comando unico de mas arriba. Este cuaderno se limita a
leer los artefactos ya producidos, de modo que es reproducible sin descargar
nada.
